# Modeling and Evaluating 

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
import os, sys, gc, time, warnings, pickle, psutil, random, optuna
from sklearn.metrics import mean_squared_error
print("Num GPUs Available:", len(tf.config.experimental.list_physical_devices('GPU')))
print("GPU Details:", tf.config.experimental.list_physical_devices('GPU'))

Num GPUs Available: 1
GPU Details: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [21]:
path = "../../data"

calendar = pd.read_csv(os.path.join(path, "calendar.csv"))
prices = pd.read_csv(os.path.join(path, "sell_prices.csv"))
sales = pd.read_csv(os.path.join(path, "sales_train_evaluation.csv"))

In [22]:
def prep_calendar(df):
    df = df.drop(["weekday", "event_type_1", "event_type_2"], axis=1)
    df = df.assign(d = df.d.str[2:].astype(int))
    to_ordinal = ["event_name_1", "event_name_2"] 
    df[to_ordinal] = df[to_ordinal].fillna("1")
    df[to_ordinal] = OrdinalEncoder(dtype="int").fit_transform(df[to_ordinal]) + 1
    to_int8 = ["wday", "month", "snap_CA", "snap_TX", "snap_WI"] + to_ordinal
    df[to_int8] = df[to_int8].astype("int8")
    
    return df

calendar = prep_calendar(calendar)
calendar.head()

,date,wm_yr_wk,wday,month,year,d,event_name_1,event_name_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,1,1,2011,1,1,1,0,0,0
1,2011-01-30,11101,2,1,2011,2,1,1,0,0,0
2,2011-01-31,11101,3,1,2011,3,1,1,0,0,0
3,2011-02-01,11101,4,2,2011,4,1,1,1,1,0
4,2011-02-02,11101,5,2,2011,5,1,1,1,0,1


In [34]:
def downcast(df):
    cols = df.dtypes.index.tolist()
    types = df.dtypes.values.tolist()
    
    for i, t in enumerate(types):
        col = cols[i]
        if 'int' in str(t):
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)

        elif 'float' in str(t):
            c_min = df[col].min()
            c_max = df[col].max()
            # ⚠️ Không dùng float16 vì không lưu được trong Parquet
            if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                df[col] = df[col].astype(np.float16)
            if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)    
    return df

In [30]:
calendar = pd.read_parquet('../data/done_cal.parquet')
prices = pd.read_parquet('../data/done_prices.parquet')
sales = pd.read_parquet('../data/done_sales.parquet')
done_dis = pd.read_parquet('../../data/done_dis.parquet')

In [31]:
print(f'{round(calendar.memory_usage(deep=True).sum()/1024**2,2)} MB')
print(f'{round(prices.memory_usage(deep=True).sum()/1024**2,2)} MB')
print(f'{round(sales.memory_usage(deep=True).sum()/1024**2,2)} MB')
print(f'{round(done_dis.memory_usage(deep=True).sum()/1024**2,2)} MB')

0.04 MB
58.98 MB
98.74 MB
0.05 MB


In [32]:
sales

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30485,FOODS_3_823_WI_3_evaluation,FOODS_3_823,FOODS_3,FOODS,WI_3,WI,0,0,2,2,...,1,0,3,0,1,1,0,0,1,1
30486,FOODS_3_824_WI_3_evaluation,FOODS_3_824,FOODS_3,FOODS,WI_3,WI,0,0,0,0,...,0,0,0,0,0,0,1,0,1,0
30487,FOODS_3_825_WI_3_evaluation,FOODS_3_825,FOODS_3,FOODS,WI_3,WI,0,6,0,2,...,0,0,1,2,0,1,0,1,0,2
30488,FOODS_3_826_WI_3_evaluation,FOODS_3_826,FOODS_3,FOODS,WI_3,WI,0,0,0,0,...,1,1,1,4,6,0,1,1,1,0


In [33]:
LAGS = [7, 28]
WINDOWS = [7, 28]
FIRST = 1942
LENGTH = 28

def demand_features(df):
    """ Derive features from sales data and remove rows with missing values """
    
    for lag in LAGS:
        df[f'lag_t{lag}'] = df.groupby('id')['demand'].transform(lambda x: x.shift(lag)).astype("float32")
        for w in WINDOWS:
            df[f'rolling_mean_lag{lag}_w{w}'] = df.groupby('id')[f'lag_t{lag}'].transform(lambda x: x.rolling(w).mean()).astype("float32")
        
    return df

def prep_data(df, drop_d=1000):
    """ Prepare model data sets """
    
    # Kick out old dates
    df = df.drop(["d_" + str(i+1) for i in range(drop_d)], axis=1)

    # Reshape to long
    df = df.assign(id=df.id.str.replace("_evaluation", ""))
    df = df.reindex(columns=df.columns.tolist() + ["d_" + str(FIRST + i) for i in range(LENGTH)])
    df = df.melt(id_vars=["id", "item_id", "store_id", "state_id", "dept_id", "cat_id"], var_name='d', value_name='demand')
    df = df.assign(d=df.d.str[2:].astype("int64"),
                   demand=df.demand.astype("float32"))
    
    # Add demand features
    df = demand_features(df)
    
    # Remove rows with NAs
    df = df[df.d > (drop_d + max(LAGS) + max(WINDOWS))]
 
    # Join calendar & prices
    df = df.merge(calendar, how="left", on="d")
    df = df.merge(prices, how="left", on=["store_id", "item_id", "wm_yr_wk"])
    df = df.drop(["wm_yr_wk"], axis=1)
    
    # Ordinal encoding of remaining categorical fields
    for v in ["item_id", "store_id", "state_id", "dept_id", "cat_id"]:
        df[v] = OrdinalEncoder(dtype="int").fit_transform(df[[v]]).astype("int16") + 1
    
    return df

df = prep_data(sales, 1000 - 28)
print(f'{round(df.memory_usage(deep=True).sum()/1024**2,2)} MB')

4097.29 MB


In [35]:
df = downcast(df)
print(f'{round(df.memory_usage(deep=True).sum()/1024**2,2)} MB')

3823.67 MB


In [36]:
df.to_parquet('../../data/test.parquet')

In [3]:
null_columns = df.columns[df.isna().any()].tolist()

# Kiểm tra giá trị 0
zero_columns = df.columns[(df == 0).any()].tolist()

print("Các cột có giá trị null:", null_columns)
print("Các cột có giá trị 0:", zero_columns)

Các cột có giá trị null: ['demand', 'lag_t7', 'rolling_mean_lag7_w7', 'rolling_mean_lag7_w28', 'sell_price', 'disaster_type']
Các cột có giá trị 0: ['demand', 'lag_t7', 'rolling_mean_lag7_w7', 'rolling_mean_lag7_w28', 'lag_t28', 'rolling_mean_lag28_w7', 'rolling_mean_lag28_w28', 'snap_CA', 'snap_TX', 'snap_WI', 'disaster_severity_flag', 'flag_ufc']


In [6]:
df = df.drop(['date','disaster_type'],axis=1)
sample_submission = pd.read_csv('../../data/sample_submission.csv')
sample_submission = downcast(sample_submission)

LAGS = [7, 28]
WINDOWS = [7, 28]
FIRST = 1942 # first to predict
LENGTH = 28

def demand_features_eval(df):
    """ Same as demand_features but for the step-by-step evaluation """
    out = df.groupby('id', sort=False).last()
    for lag in LAGS:
        out[f'lag_t{lag}'] = df.groupby('id', sort=False)['demand'].nth(-lag-1).astype("float32")
        for w in WINDOWS:
            out[f'rolling_mean_lag{lag}_w{w}'] = df.groupby('id', sort=False)['demand'].nth(list(range(-lag-w, -lag))).groupby('id', sort=False).mean().astype("float32")
    
    return out.reset_index()

def prep_train(df,drop_d = 1000):

    x = list(set(df.columns) - {'id', 'd', 'demand'}) # exclude 'date' from features
    print(x)
    # Split into test, valid, train
    test = df[df.d >= FIRST - max(LAGS) - max(WINDOWS) - 28]
    df = df[df.d < FIRST]

    xtrain, xvalid, ytrain, yvalid = train_test_split(df[x], df["demand"], test_size=0.1, shuffle=True, random_state=54)
    train = lgb.Dataset(xtrain, label = ytrain)
    valid = lgb.Dataset(xvalid, label = yvalid)

    # return validation data for Optuna
    return train, valid, test, x, xvalid, yvalid



def pred_all(fit, test, x):
    """ Calculate predictions """
    
    # Recursive prediction
    for i, day in enumerate(np.arange(FIRST, FIRST + LENGTH)):
        test_day = demand_features_eval(test[(test.d <= day) & (test.d >= day - max(LAGS) - max(WINDOWS))])
        test.loc[test.d == day, "demand"] = fit.predict(test_day[x])
    
    return test

def pred_to_csv(test, cols=sample_submission.columns, file="submission.csv"):
    """ Reshape predictions and save submission csv """
     
    # Prepare for reshaping
    test = test.assign(id=test.id + "_" + np.where(test.d < FIRST, "validation", "evaluation"),
                       F="F" + (test.d - FIRST + LENGTH + 1 - LENGTH * (test.d >= FIRST)).astype("str"))
    
    # Reshape
    submission = test.pivot(index="id", columns="F", values="demand").reset_index()[cols].fillna(1)
    
    # Export
    submission.to_csv(file, index=False)
    
    return True

train, valid, test, x, X_valid, y_valid = prep_train(df, 1000 - 28)

del df
gc.collect()

['year', 'rolling_mean_lag28_w28', 'snap_CA', 'rolling_mean_lag28_w7', 'lag_t28', 'state_id', 'sell_price', 'disaster_severity_flag', 'dept_id', 'flag_ufc', 'lag_t7', 'snap_TX', 'wday', 'rolling_mean_lag7_w7', 'snap_WI', 'event_name_1', 'rolling_mean_lag7_w28', 'cat_id', 'humidity', 'gold_price', 'event_name_2', 'item_id', 'store_id', 'month']


0

In [7]:
def objective(trial):
    # Gợi ý các hyperparameter
    param = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'feature_pre_filter': False,
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'learning_rate': trial.suggest_float('learning_rate', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 1000, 3000),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }

    callbacks = [
        early_stopping(stopping_rounds=200),
        log_evaluation(period=100)
    ]

    # Train
    gbm = lgb.train(
        param,
        train,
        valid_sets=[valid],
        num_boost_round=600,    
        callbacks=callbacks,
    )

    preds = gbm.predict(X_valid)
    rmse = np.sqrt(mean_squared_error(y_valid, preds))

    return rmse  # Tối ưu RMSE → minimize

# Study
study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=50)

# Kết quả
print("Best trial:")
trial = study.best_trial
print(f"  RMSE: {trial.value}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

[I 2025-05-06 23:19:01,198] A new study created in memory with name: no-name-b66abaa7-e182-49d9-abe9-f32552557715


Training until validation scores don't improve for 200 rounds
[100]	valid_0's rmse: 3.66244
[200]	valid_0's rmse: 3.65918
[300]	valid_0's rmse: 3.65593
[400]	valid_0's rmse: 3.65268
[500]	valid_0's rmse: 3.64944
[600]	valid_0's rmse: 3.64621
Did not meet early stopping. Best iteration is:
[600]	valid_0's rmse: 3.64621


[I 2025-05-06 23:49:40,867] Trial 0 finished with value: 3.6462059984437865 and parameters: {'lambda_l1': 0.29664749239867805, 'learning_rate': 1.4199622753562034e-05, 'lambda_l2': 5.3060609988022824e-08, 'num_leaves': 2746, 'feature_fraction': 0.9400904329725311, 'bagging_fraction': 0.9502639359496837, 'bagging_freq': 3, 'min_child_samples': 47}. Best is trial 0 with value: 3.6462059984437865.


Training until validation scores don't improve for 200 rounds
[100]	valid_0's rmse: 2.13518
[200]	valid_0's rmse: 2.13559
[300]	valid_0's rmse: 2.13697
Early stopping, best iteration is:
[108]	valid_0's rmse: 2.13344


[I 2025-05-07 00:03:29,550] Trial 1 finished with value: 2.1334440342018963 and parameters: {'lambda_l1': 0.6450856055966956, 'learning_rate': 0.21245830694316561, 'lambda_l2': 0.5158666199278924, 'num_leaves': 1293, 'feature_fraction': 0.7058005190772199, 'bagging_fraction': 0.6693136939418065, 'bagging_freq': 6, 'min_child_samples': 84}. Best is trial 1 with value: 2.1334440342018963.


Training until validation scores don't improve for 200 rounds
[100]	valid_0's rmse: 3.6559
[200]	valid_0's rmse: 3.64615
[300]	valid_0's rmse: 3.63645
[400]	valid_0's rmse: 3.62683
[500]	valid_0's rmse: 3.61725
[600]	valid_0's rmse: 3.60773
Did not meet early stopping. Best iteration is:
[600]	valid_0's rmse: 3.60773


... (92 outputs left out)

[I 2025-05-07 06:08:52,669] Trial 48 finished with value: 2.0920525384977195 and parameters: {'lambda_l1': 0.0017619212704032057, 'learning_rate': 0.0834123934657015, 'lambda_l2': 1.013511925034488e-07, 'num_leaves': 1565, 'feature_fraction': 0.4203365523181927, 'bagging_fraction': 0.6049427358584042, 'bagging_freq': 5, 'min_child_samples': 18}. Best is trial 31 with value: 2.062165561921754.


Training until validation scores don't improve for 200 rounds
[100]	valid_0's rmse: 3.66517
[200]	valid_0's rmse: 3.66464
[300]	valid_0's rmse: 3.66411
[400]	valid_0's rmse: 3.66357
[500]	valid_0's rmse: 3.66304
[600]	valid_0's rmse: 3.66251
Did not meet early stopping. Best iteration is:
[600]	valid_0's rmse: 3.66251


[I 2025-05-07 06:20:59,398] Trial 49 finished with value: 3.66251038979607 and parameters: {'lambda_l1': 0.040263140898074666, 'learning_rate': 2.326990255292934e-06, 'lambda_l2': 0.00016461460532124072, 'num_leaves': 1730, 'feature_fraction': 0.8765262111233953, 'bagging_fraction': 0.7945676706555298, 'bagging_freq': 6, 'min_child_samples': 39}. Best is trial 31 with value: 2.062165561921754.


Best trial:
  RMSE: 2.062165561921754
  Params: 
    lambda_l1: 0.002354253988281785
    learning_rate: 0.05303667024743503
    lambda_l2: 7.053121220857246e-05
    num_leaves: 1601
    feature_fraction: 0.9834112644063673
    bagging_fraction: 0.9606769151362458
    bagging_freq: 2
    min_child_samples: 30
